# Test IRIS Telescope Integration (Mixed Protocols)

This notebook is used for interactive testing of the new `TreeIrisObservatory` implementation and the `Pilar` and `IrisCCD` connectors.

**What does this notebook do?**
1. Runs local Mock servers in the background (TCP for Pilar, UDP for IrisCCD).
2. Replaces the configuration to connect to `localhost` instead of real hardware.
3. Initializes the IRIS device tree.
4. Allows executing `get` and `put` commands directly from the cells.

In [1]:
import asyncio
import logging
import sys
import os
from unittest.mock import MagicMock

# Add the path to the main project directory to see the obsrv modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from obsrv.tree_components.specialized_components.tree_iris import TreeIrisObservatory
from obsrv.ob_config import SingletonConfig

# Logging configuration (to see what is happening in the connectors)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("JupyterIris")

## 1. Hardware Mocks Definition
The following classes simulate the operation of physical hardware (Pilar Telescope and UDP Camera).

In [2]:
class MockPilarServer:
    """A simple TCP server mocking the Pilar telescope."""
    def __init__(self, port=5950):
        self.server = None
        self.host = '127.0.0.1'
        self.port = port
        self.responses = {
            "GET OBJECT.EQUATORIAL.RA": ("OBJECT.EQUATORIAL.RA=10.5", True),
            "GET OBJECT.EQUATORIAL.DEC": ("OBJECT.EQUATORIAL.DEC=-20.0", True),
            "SET POINTING.SETUP.FOCUS.POSITION=2000": ("POINTING.SETUP.FOCUS.POSITION=2000", True),
        }

    async def handle_client(self, reader, writer):
        try:
            while True:
                data = await reader.readline()
                if not data: break
                line = data.decode().strip()
                if not line: continue
                
                parts = line.split(' ', 1)
                if len(parts) < 2: continue
                cmd_id, command = parts[0], parts[1]
                
                # Response logic
                if command in self.responses:
                    val, ok = self.responses[command]
                    writer.write(f"{cmd_id} {val}\n".encode())
                    status = "COMMAND COMPLETE" if ok else "COMMAND FAILED"
                    writer.write(f"{cmd_id} {status}\n".encode())
                else:
                    writer.write(f"{cmd_id} COMMAND COMPLETE\n".encode())
                await writer.drain()
        except Exception:
            pass

    async def start(self):
        self.server = await asyncio.start_server(self.handle_client, self.host, self.port)
        print(f"[MockPilar] Listening on TCP {self.host}:{self.port}")
        asyncio.create_task(self.server.serve_forever())

class MockIrisCcdProtocol(asyncio.DatagramProtocol):
    def __init__(self):
        self.transport = None

    def connection_made(self, transport):
        # Here the protocol automatically receives the transport from asyncio
        self.transport = transport

    def datagram_received(self, data, addr):
        msg = data.decode().strip()
        resp = "**** OKAY 0" if msg == "sync" else "**** OKAY 1"
        if "temp" in msg: resp = "**** OKAY -15.5"
        
        if self.transport:
            self.transport.sendto(resp.encode(), addr)
        
class MockIrisCcdServer:
    """A simple UDP server mocking the camera."""
    def __init__(self):
        self.transport = None
        self.protocol = None

    async def start(self, port=8888):
        loop = asyncio.get_running_loop()
        # create_datagram_endpoint returns (transport, protocol)
        self.transport, self.protocol = await loop.create_datagram_endpoint(
            lambda: MockIrisCcdProtocol(), local_addr=('127.0.0.1', port)
        )
        print(f"[MockIris] Listening on UDP 127.0.0.1:{port}")

## 2. Mock Startup and Configuration
We start the servers in the background and replace `SingletonConfig` so that IRIS points to `localhost`.

In [3]:
import confuse
from unittest.mock import MagicMock
import asyncio
import logging

# --- BULLETPROOF MockCfg CLASS ---
class MockCfg:
    """
    Mocks the behavior of the confuse library.
    Does not throw errors on [] access, only on .get().
    """
    def __init__(self, value, exists=True, path="root"):
        self.value = value
        self._exists = exists
        self._path = path

    def __getitem__(self, key):
        # Confuse returns a View object even if the key does not exist
        new_path = f"{self._path}.{key}"
        if not self._exists:
            return MockCfg(None, exists=False, path=new_path)
        
        if isinstance(self.value, dict):
            if key not in self.value:
                # Key is missing, but we return an object (marked as non-existent)
                return MockCfg(None, exists=False, path=new_path)
            return MockCfg(self.value[key], path=new_path)
        
        # Attempt to go deeper into a value that is not a dictionary
        return MockCfg(None, exists=False, path=new_path)

    def get(self, template=None):
        if not self._exists:
            # Here we throw a specific confuse error that the code can catch
            raise confuse.exceptions.NotFoundError(f"Mock config not found: {self._path}")
        return self.value

    # Iteration methods (e.g., for component in config['components'])
    def keys(self):
        if self._exists and isinstance(self.value, dict):
            return self.value.keys()
        return []

    def __iter__(self):
        if self._exists and isinstance(self.value, dict):
            return iter(self.value)
        return iter([])

    def __contains__(self, item):
        if self._exists and isinstance(self.value, dict):
            return item in self.value
        return False
        
    def as_str_seq(self):
        # Emulate getting a list of strings (e.g., flags)
        val = self.get()
        if isinstance(val, list):
            return [str(v) for v in val]
        return []

# --- Start servers (protection against port in use error) ---
try:
    pilar_mock = MockPilarServer()
    iris_mock = MockIrisCcdServer()
    await pilar_mock.start()
    await iris_mock.start()
except OSError:
    print("Servers are probably already running (ports in use). Continuing.")

# --- Full Test Configuration ---
# Complemented with fields that Observatory might look for (flags, coords, etc.)
test_config_raw = {
    'tree': {
        'iris-notebook': {
            'timeout_multiplier': 0.8
        },
        'iris': {
            'observatory': {
                'comment': 'Test IRIS Telescope',
                'flags': ['production'],
                'lon': -70.2,
                'lat': -24.6,
                'elev': 2800,
                'epoch': 2000,
                'style': {'color': '#FF0000'},
                'protocol': 'ignored', 
                'components': {
                    'mount': {
                        'kind': 'telescope', 'device_number': 0, 
                        'protocol': 'pilar', 'address': '127.0.0.1:5950', 
                        'type': 'az', 'slew_timeout': 5
                    },
                    'focuser': {
                        'kind': 'focuser', 'device_number': 0, 
                        'protocol': 'pilar', 'address': '127.0.0.1:5950'
                    },
                    'camera': {
                        'kind': 'camera', 'device_number': 0, 
                        'protocol': 'iris_ccd', 'address': '127.0.0.1:8888'
                    },
                    'dome': {
                        'kind': 'dome', 'device_number': 0, 
                        'protocol': 'alpaca', 'address': 'http://127.0.0.1:11111/api/v1',
                        'slew_timeout': 10
                    }
                }
            }
        }
    }
}

# Wrap in MockCfg
mock_conf_obj = MockCfg(test_config_raw)

# Patch SingletonConfig
SingletonConfig.get_config = MagicMock(return_value=mock_conf_obj)
print("Configuration updated. MockCfg is ready for anything.")

[MockPilar] Listening on TCP 127.0.0.1:5950
[MockIris] Listening on UDP 127.0.0.1:8888
Configuration updated. MockCfg is ready for anything.


## 3. IRIS Tree Initialization
We create the `TreeIrisObservatory` object. At this point, the connectors are being prepared (lazy loading).

In [4]:
tree_iris = TreeIrisObservatory('iris-notebook', observatory_name='iris')
print("IRIS tree initialized. Available components:")
for child in tree_iris._observatory.children:
    print(f" - {child}")

2026-01-21 13:22:17,712 - tree_iris - WARNING - Could not load configuration for iris: Unknown protocol: ignored. Available: ['alpaca', 'pilar', 'beso', 'iris_ccd', 'dummy']


IRIS tree initialized. Available components:


## 4. Pilar Protocol Tests (Telescope)
We fetch `rightascension` and test query parallelism.

In [5]:
# Simple fetch (we use children['mount'] instead of .mount)
# Depending on the Observatory implementation, access might be via children['mount']
mount = tree_iris._observatory.children['mount']
ra = await mount.get('rightascension')
print(f"Right Ascension: {ra} degrees (Mock returns 10.5h)")

KeyError: 'mount'

In [ ]:
# Parallelism test
mount = tree_iris._observatory.children['mount']
tasks = [
    mount.get('rightascension'),
    mount.get('declination'),
    mount.get('rightascension'),
    mount.get('declination'),
    mount.get('rightascension')
]
results = await asyncio.gather(*tasks)
print(f"Parallel query results: {results}")

# PUT test (focuser movement)
focuser = tree_iris._observatory.children['focuser']
resp = await focuser.put('move', Position=2000)
print(f"Focuser response: {resp}")

In [ ]:
# PUT test (focuser movement)
resp = await tree_iris._observatory.focuser.put('move', Position=2000)
print(f"Focuser response: {resp}")

## 5. IrisCCD Protocol Tests (Camera)
We test UDP communication.

In [ ]:
# Fetch camera status
camera = tree_iris._observatory.children['camera']
state = await camera.get('camerastate')
print(f"Camera State (UDP): {state}")

# Fetch temperature (simulated)
temp = await camera.get('ccdtemperature')
print(f"Camera Temp: {temp}")

## 6. Cleanup
Stopping servers after tests.

In [ ]:
pilar_mock.server.close()
await pilar_mock.server.wait_closed()
iris_mock.transport.close()
print("Mocks stopped.")